In [7]:
from statsbombpy import sb
import sys
sys.path.insert(0, '..')   # so Jupyter can find your src/ folder

from src.chains import build_corpus, corpus_stats

import warnings
from statsbombpy.api_client import NoAuthWarning

warnings.filterwarnings("ignore", category=NoAuthWarning)

# Load ONE match to test (La Liga 2015/16)
matches = sb.matches(competition_id=11, season_id=26)
test_match_id = matches.iloc[0]['match_id']

events = sb.events(match_id=test_match_id)
print(f"Events in match: {len(events)}")
print(f"Columns: {list(events.columns)}")

from statsbombpy import sb
from src.tokenise import tokenise_event

# Load one match
matches = sb.matches(competition_id=11, season_id=26)
events  = sb.events(match_id=matches.iloc[0]['match_id'])

# Test tokenisation
results = []
for _, row in events.iterrows():
    token = tokenise_event(row.to_dict())
    if token:
        results.append(token)

from collections import Counter
print(Counter(results))

Events in match: 3823
Columns: ['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'block_deflection', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_no_touch', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass

In [8]:
from importlib import reload
import src.tokenise
reload(src.tokenise)          # force Python to reload the saved file
from src.tokenise import tokenise_event

from statsbombpy import sb
from collections import Counter

matches = sb.matches(competition_id=11, season_id=26)
events  = sb.events(match_id=matches.iloc[0]['match_id'])

counter = Counter()
for _, row in events.iterrows():
    token = tokenise_event(row.to_dict())
    if token:
        counter[token] += 1

print("Token counts:")
for token in ['PASS_PROG','PASS_BACK','PASS_CROSS','CARRY',
              'DRIBBLE_W','DRIBBLE_L','SHOT_ON','SHOT_OFF',
              'PRESS','INT','CLEARANCE','FOUL_W', 'CARRY_WIDE_PROG', 'PASS_CROSS', 'SHOT_ON_BOX', 'SHOT_OFF_BOX']:
    count = counter.get(token, 0)
    status = "YES" if count > 0 else " MISSING"
    print(f"  {token:<15} {count:>4}   {status}")

Token counts:
  PASS_PROG        309   YES
  PASS_BACK         22   YES
  PASS_CROSS        14   YES
  CARRY            378   YES
  DRIBBLE_W         31   YES
  DRIBBLE_L         29   YES
  SHOT_ON            2   YES
  SHOT_OFF           6   YES
  PRESS            397   YES
  INT                8   YES
  CLEARANCE         32   YES
  FOUL_W            27   YES
  CARRY_WIDE_PROG  108   YES
  PASS_CROSS        14   YES
  SHOT_ON_BOX        7   YES
  SHOT_OFF_BOX      13   YES


In [ ]:
# debug!! to check the structure of the data
from statsbombpy import sb

matches = sb.matches(competition_id=11, season_id=26)
events  = sb.events(match_id=matches.iloc[0]['match_id'])

shots = events[events['type'] == 'Shot']
print(f"Shots in match: {len(shots)}")
if len(shots) > 0:
    row = shots.iloc[0]
    print(f"\nshot_outcome value:  {repr(row['shot_outcome'])}")
    print(f"shot_outcome type:   {type(row['shot_outcome'])}")

dribbles = events[events['type'] == 'Dribble']
print(f"\nDribbles in match: {len(dribbles)}")
if len(dribbles) > 0:
    row = dribbles.iloc[0]
    print(f"\ndribble_outcome value: {repr(row['dribble_outcome'])}")
    print(f"dribble_outcome type:  {type(row['dribble_outcome'])}")

passes = events[events['type'] == 'Pass'].head(5)
for _, row in passes.iterrows():
    print(f"\npass_cross: {repr(row['pass_cross'])}  |  location: {row['location']}  |  end: {row['pass_end_location']}")

Shots in match: 30

shot_outcome value:  'Off T'
shot_outcome type:   <class 'str'>

Dribbles in match: 60

dribble_outcome value: 'Complete'
dribble_outcome type:  <class 'str'>

pass_cross: nan  |  location: [61.0, 40.1]  |  end: [57.6, 36.8]

pass_cross: nan  |  location: [55.3, 37.5]  |  end: [43.9, 41.2]

pass_cross: nan  |  location: [43.9, 41.2]  |  end: [43.9, 15.2]

pass_cross: nan  |  location: [43.9, 15.2]  |  end: [30.7, 30.9]

pass_cross: nan  |  location: [34.3, 26.7]  |  end: [27.2, 51.7]


In [10]:
corpus, player_actions = build_corpus(events)
corpus_stats(corpus)

print("\nSample possession chains:")
for chain in corpus[10:15]:
    print(chain)

Total chains:       344
Average chain len:  4.7 tokens
Longest chain:      26 tokens
Shortest chain:     2 tokens

Token frequencies:
  CARRY                368
  PRESS                353
  CARRY_WIDE           284
  PASS_PROG            240
  CARRY_PROG           114
  CARRY_WIDE_PROG      100
  DRIBBLE_W             28
  CLEARANCE             24
  FOUL_W                24
  PASS_BACK             20
  DRIBBLE_L             19
  PASS_CROSS            14
  SHOT_OFF_BOX          12
  SHOT_ON_BOX            5
  SHOT_OFF               5
  INT                    4
  SHOT_ON                1

Sample possession chains:
['PASS_PROG', 'PASS_PROG']
['PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG']
['PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG']
['PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG']
['PASS_PROG', 'PASS_PROG', 'PASS_BACK', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_PROG', 'PASS_CROSS']


- 1 - 2017/2018 
- 2 - 2016/2017
- 4 - 2018/2019 
- 90 - 2020/2021 
- 26 - 2014/2015 
- 27 - 2015/2016 
- 37 - 2012/2013 
- 38 - 2013/2014 
- 42 - 2019/2020 

In [11]:
import pickle

season_ids = [1, 2, 4, 90, 26, 27, 37, 38, 42]   
all_corpus = []
all_player_actions = {}

for sid in season_ids:
    matches = sb.matches(competition_id=11, season_id=sid)
    print(f"\nSeason {sid}: {len(matches)} matches")

    for i, (_, match) in enumerate(matches.iterrows()):
        events = sb.events(match_id=match['match_id'])
        corpus, pa = build_corpus(events)
        all_corpus.extend(corpus)

        for pid, tokens in pa.items():
            if pid not in all_player_actions:
                all_player_actions[pid] = []
            all_player_actions[pid].extend(tokens)

        if (i+1) % 20 == 0:
            print(f"  {i+1}/{len(matches)} matches done...")

    print(f"  Corpus size so far: {len(all_corpus):,} chains")

# Save both files
with open('../data/processed/corpus.pkl', 'wb') as f:
    pickle.dump(all_corpus, f)

with open('../data/processed/player_actions.pkl', 'wb') as f:
    pickle.dump(all_player_actions, f)

print(f"\nDone! Saved {len(all_corpus):,} chains for {len(all_player_actions):,} players")


Season 1: 36 matches
  20/36 matches done...
  Corpus size so far: 10,991 chains

Season 2: 34 matches
  20/34 matches done...
  Corpus size so far: 21,186 chains

Season 4: 34 matches
  20/34 matches done...
  Corpus size so far: 31,349 chains

Season 90: 35 matches
  20/35 matches done...
  Corpus size so far: 41,257 chains

Season 26: 38 matches
  20/38 matches done...
  Corpus size so far: 52,769 chains

Season 27: 380 matches
  20/380 matches done...
  40/380 matches done...
  60/380 matches done...
  80/380 matches done...
  100/380 matches done...
  120/380 matches done...
  140/380 matches done...
  160/380 matches done...
  180/380 matches done...
  200/380 matches done...
  220/380 matches done...
  240/380 matches done...
  260/380 matches done...
  280/380 matches done...
  300/380 matches done...
  320/380 matches done...
  340/380 matches done...
  360/380 matches done...
  380/380 matches done...
  Corpus size so far: 170,180 chains

Season 37: 7 matches
  Corpus size s